Considere o cenário onde se tem 6 carros para serem transportados em uma balsa. Cada carro tem um valor de frete e um peso.
- Carro 1: frete de R\$100 e peso de 50kg;
- Carro 2: frete de R\$50 e peso de 80kg;
- Carro 3: frete de R\$200 e peso de 35kg;
- Carro 4: frete de R\$70 e peso de 70kg;
- Carro 5: frete de R\$180 e peso de 110kg;
- Carro 6: frete de R\$80 e peso de 90kg.

A balsa suporta até 250kg e o transportador deseja maximizar o lucro por meio do frete. Obviamente, carros ficarão de fora. Modele o problema.

In [1]:
!pip install ortools

  Using cached ortools-9.15.6755-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached ortools-9.15.6755-cp313-cp313-win_amd64.whl (23.9 MB)
Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl (437 kB)
Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.3 MB 10.3 MB/s eta 0:00:02
   ------------- -------------------------- 4.2/12.3 MB 13.9 MB/s eta 0:00:01
   ----------------- ---------------------- 5.2/12.3 MB 11.9 MB/s eta 0:00:01
   --------------------------- ------------ 8.4/12.3 MB 11.4 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 12.0 M

In [2]:
from ortools.linear_solver import pywraplp

In [3]:
n = 6
peso_max = 250
fretes = [100, 50, 200, 70, 180, 80]
pesos = [50, 80, 35, 70, 110, 90]
carros = [f'Carro {i + 1}' for i in range(n)]

In [4]:
solver = pywraplp.Solver.CreateSolver('SCIP')
if solver is None:
    raise RuntimeError('Nao foi possivel criar o solver SCIP.')

infinity = solver.infinity()

In [5]:
x = {i: solver.BoolVar(f'x{i}') for i in range(n)}

In [6]:
objetivo = solver.Objective()
for i in range(n):
    objetivo.SetCoefficient(x[i], fretes[i])
objetivo.SetMaximization()

In [7]:
restricao = solver.RowConstraint(-infinity, peso_max, 'peso')
for i in range(n):
    restricao.SetCoefficient(x[i], pesos[i])

In [8]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 1
\   Variables        : 6
\     Binary         : 6
\     Integer        : 0
\     Continuous     : 0
Maximize
 Obj: +100 x0 +50 x1 +200 x2 +70 x3 +180 x4 +80 x5 
Subject to
 peso: +50 x0 +80 x1 +35 x2 +70 x3 +110 x4 +90 x5  <= 250
Bounds
 0 <= x0 <= 1
 0 <= x1 <= 1
 0 <= x2 <= 1
 0 <= x3 <= 1
 0 <= x4 <= 1
 0 <= x5 <= 1
Binaries
 x0
 x1
 x2
 x3
 x4
 x5
End



In [9]:
status = solver.Solve()

In [10]:
if status == pywraplp.Solver.OPTIMAL:
    print('Lucro máximo:', objetivo.Value())
    print('Peso total:', sum(pesos[i] * x[i].solution_value() for i in range(n)))
    print('Carros selecionados:')
    for i in range(n):
        if x[i].solution_value() > 0.5:
            print(f'- {carros[i]}: frete R${fretes[i]} e peso {pesos[i]}kg')
else:
    print('Modelo sem solução ótima.')

Lucro máximo: 480.0
Peso total: 195.0
Carros selecionados:
- Carro 1: frete R$100 e peso 50kg
- Carro 3: frete R$200 e peso 35kg
- Carro 5: frete R$180 e peso 110kg
